# Baby Step 6 — Internal Transaction Design and Valuation Sensitivities

This notebook converts VoltEdge's synthetic $32m base / $43m downside funding range into comparable capital structures and five-year valuation and investor-return sensitivities.

It is an internal architecture demonstration—not investment advice, a valuation opinion, a financing commitment or authority to contact a counterparty.


## The control question

Which internal structure best balances funding certainty, dilution, leverage, negative free cash flow and downside protection?

The notebook compares:

1. Common equity.
2. Staged nonparticipating preferred equity.
3. Growth debt.
4. Blended common equity and debt.
5. Preferred equity plus a downside delayed draw.

It defaults to **dry run** and does not modify the vault.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import sys

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value): print(value)
    def Markdown(value): return value

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DEFAULT_COLAB_VAULT = Path("/content/drive/MyDrive/Alejandro-Reynoso-Investment-Banking-Vault")
LOCAL_VAULT = Path("/workspace/scratch/9ba1ff46ede5/Alejandro-Reynoso-Investment-Banking-Vault")
VAULT = Path(os.environ.get("IB_VAULT_PATH", DEFAULT_COLAB_VAULT if RUNNING_IN_COLAB else LOCAL_VAULT))

WRITE_REPRODUCTION_ARTIFACTS = False

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"Vault: {VAULT}")
print(f"Dry run: {not WRITE_REPRODUCTION_ARTIFACTS}")


## 1. Resolve and validate the Step 6 inputs


In [ ]:
required = [
    VAULT / "Data" / "company_master.csv",
    VAULT / "Data" / "claim_register.csv",
    VAULT / "Data" / "source_registry.csv",
    VAULT / "Data" / "loop_006_transaction_structures.csv",
    VAULT / "Data" / "loop_006_valuation_sensitivities.csv",
    VAULT / "Data" / "loop_006_investor_returns.csv",
    VAULT / "Data" / "loop_006_evidence_scores.csv",
    VAULT / "Data" / "validation_report.json",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing required files:\n" + "\n".join(missing)

validation = json.loads((VAULT / "Data" / "validation_report.json").read_text())
assert validation["companies"] == 102
assert validation["unresolved_wikilinks"] == []
assert validation["canvas_errors"] == []
display(pd.Series({
    "companies": validation["companies"],
    "markdown_notes": validation["markdown_notes"],
    "unresolved_wikilinks": len(validation["unresolved_wikilinks"]),
    "canvas_errors": len(validation["canvas_errors"]),
}, name="value").to_frame())


## 2. Load the governed transaction-design layer


In [ ]:
companies = pd.read_csv(VAULT / "Data" / "company_master.csv")
claims = pd.read_csv(VAULT / "Data" / "claim_register.csv")
sources = pd.read_csv(VAULT / "Data" / "source_registry.csv")
structures = pd.read_csv(VAULT / "Data" / "loop_006_transaction_structures.csv")
valuations = pd.read_csv(VAULT / "Data" / "loop_006_valuation_sensitivities.csv")
returns = pd.read_csv(VAULT / "Data" / "loop_006_investor_returns.csv")
evidence = pd.read_csv(VAULT / "Data" / "loop_006_evidence_scores.csv")

assert len(sources) == 9 and sources["id"].is_unique
assert len(claims) == 18 and claims["id"].is_unique
assert len(structures) == 5 and structures["option_id"].is_unique
assert len(valuations) == 3 and len(returns) == 3
assert round(float(evidence["confidence_score"].mean()), 1) == 75.2
print(f"Loaded {len(structures)} structures, {len(valuations)} valuation cases and {len(claims)} governed claims.")


## 3. Reconstruct the entry capitalization


In [ ]:
volt = companies.loc[companies["name"].eq("VoltEdge Thermal Systems")].iloc[0]
enterprise_value = float(volt["enterprise_value_usd_m"])
debt = float(volt["debt_usd_m"])
cash = float(volt["cash_usd_m"])
ebitda = float(volt["ebitda_usd_m"])
revenue = float(volt["revenue_usd_m"])
pre_money_equity = enterprise_value - debt + cash
preferred_investment = 32.0
ownership_if_converted = preferred_investment / (pre_money_equity + preferred_investment)

assert pre_money_equity == 413.0
assert round(ownership_if_converted * 100, 2) == 7.19
display(pd.Series({
    "enterprise_value_usd_m": enterprise_value,
    "debt_usd_m": debt,
    "cash_usd_m": cash,
    "pre_money_equity_usd_m": pre_money_equity,
    "preferred_investment_usd_m": preferred_investment,
    "ownership_if_converted_pct": ownership_if_converted * 100,
}, name="value").to_frame().round(2))


## 4. Compare the five funding alternatives


In [ ]:
recalculated_leverage = structures["gross_debt_usd_m"] / ebitda
assert (recalculated_leverage.sub(structures["pro_forma_gross_leverage_x"]).abs() < 0.01).all()

view = structures[[
    "option_id", "structure", "total_funding_usd_m", "dilution_if_converted_pct",
    "cash_interest_pct", "pik_pct", "pro_forma_gross_leverage_x", "status"
]]
display(view)

ax = structures.plot.bar(x="structure", y="pro_forma_gross_leverage_x", figsize=(10, 4), legend=False, color="#3D8DFF")
ax.axhline(4.0, color="#C83E4D", linestyle="--", label="Internal 4.0x review line")
ax.set_ylabel("Pro forma gross leverage / x"); ax.set_xlabel(""); plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()


## 5. Apply the leverage and cash-burden gate


In [ ]:
growth_debt = structures.loc[structures["option_id"].eq("STR-003")].iloc[0]
growth_debt_cash_interest = growth_debt["debt_component_usd_m"] * growth_debt["cash_interest_pct"] / 100
growth_debt_rejected = growth_debt["pro_forma_gross_leverage_x"] > 4.0 and growth_debt_cash_interest >= 3.0

assert round(growth_debt_cash_interest, 1) == 3.2
assert growth_debt_rejected
print(f"Growth debt cash interest: ${growth_debt_cash_interest:.1f}m per year")
print("Growth debt rejected:", growth_debt_rejected)


## 6. Reconstruct the recommended staged preferred structure


In [ ]:
preferred = structures.loc[structures["option_id"].eq("STR-002")].iloc[0]
preferred_checks = {
    "total_is_32": preferred["total_funding_usd_m"] == 32,
    "tranches_reconcile": preferred["upfront_funding_usd_m"] + preferred["milestone_tranche_usd_m"] == 32,
    "no_cash_interest": preferred["cash_interest_pct"] == 0,
    "pik_is_8": preferred["pik_pct"] == 8,
    "preference_is_1x": preferred["liquidation_preference_x"] == 1,
    "leverage_below_4x": preferred["pro_forma_gross_leverage_x"] < 4,
}
assert all(preferred_checks.values())
display(pd.DataFrame([preferred_checks]).T.rename(columns={0: "passed"}))
display(preferred.to_frame("value"))


## 7. Recalculate the five-year valuation sensitivities


In [ ]:
for idx, row in valuations.iterrows():
    recomputed_revenue = revenue * (1 + row["revenue_cagr_pct"] / 100) ** 5
    recomputed_ev = recomputed_revenue * row["exit_ev_revenue_x"]
    recomputed_equity = recomputed_ev - row["exit_net_debt_usd_m"]
    assert abs(recomputed_revenue - row["year_5_revenue_usd_m"]) < 0.02
    assert abs(recomputed_ev - row["implied_exit_ev_usd_m"]) < 0.02
    assert abs(recomputed_equity - row["implied_exit_equity_usd_m"]) < 0.02

display(valuations)
ax = valuations.plot.bar(x="scenario", y="implied_exit_equity_usd_m", figsize=(7, 4), legend=False, color=["#F1B44C", "#3D8DFF", "#167D58"])
ax.set_ylabel("Implied exit equity value / USDm"); ax.set_xlabel("")
plt.tight_layout(); plt.show()


## 8. Recalculate the investor preference-versus-conversion election


In [ ]:
preference_value = preferred_investment * (1 + preferred["pik_pct"] / 100) ** 5
assert round(preference_value, 2) == 47.02

recomputed = []
for _, row in valuations.iterrows():
    conversion_value = row["implied_exit_equity_usd_m"] * ownership_if_converted
    proceeds = max(preference_value, conversion_value)
    moic = proceeds / preferred_investment
    irr = moic ** (1 / 5) - 1
    election = "Convert" if conversion_value > preference_value else "Preference"
    recomputed.append([row["scenario"], proceeds, moic, irr * 100, election])

recomputed = pd.DataFrame(recomputed, columns=["scenario", "proceeds_calc", "moic_calc", "irr_pct_calc", "election_calc"])
check = returns.merge(recomputed, on="scenario")
assert (check["moic_x"].sub(check["moic_calc"]).abs() < 0.01).all()
assert (check["irr_pct"].sub(check["irr_pct_calc"]).abs() < 0.1).all()
assert (check["election"] == check["election_calc"]).all()
display(returns)


## 9. Run the Step 6 decision engine


In [ ]:
base_return = returns.loc[returns["scenario"].eq("Base")].iloc[0]
downside_return = returns.loc[returns["scenario"].eq("Downside")].iloc[0]
decision_checks = {
    "vault_valid": not validation["unresolved_wikilinks"] and not validation["canvas_errors"],
    "five_structures_compared": len(structures) == 5,
    "growth_debt_rejected": growth_debt_rejected,
    "preferred_terms_reconcile": all(preferred_checks.values()),
    "base_return_at_least_2x": base_return["moic_x"] >= 2.0,
    "downside_has_preference_floor": downside_return["election"] == "Preference" and downside_return["irr_pct"] >= 8.0,
    "model_claims_governed": set(["CLM-015", "CLM-016", "CLM-017", "CLM-018"]).issubset(set(claims["id"])),
}

recommendation = "APPROVE PREFERRED EQUITY AS INTERNAL WORKING STRUCTURE" if all(decision_checks.values()) else "HOLD"
next_authority = "AUTHORIZE SYNTHETIC COUNTERPARTY RANKING — NO OUTREACH" if recommendation.startswith("APPROVE") else "NO ADVANCEMENT"
guardrails = [
    "No company, investor, buyer or lender outreach",
    "No communicated valuation or financing terms",
    "No mandate representation or execution",
    "The $11m delayed draw requires a new gate",
    "Return with counterparty fit, conflicts and ranking",
]

display(pd.DataFrame([decision_checks]).T.rename(columns={0: "passed"}))
display(Markdown(f"### {recommendation}\n\n**Next authority:** {next_authority}\n\n" + "\n".join(f"- {g}" for g in guardrails)))
assert recommendation == "APPROVE PREFERRED EQUITY AS INTERNAL WORKING STRUCTURE"


## 10. Define the Baby Step 7 input contract


In [ ]:
next_product = pd.DataFrame([
    ["Financial investors", 12, "Check size, preferred-equity appetite, target return, governance and conflicts"],
    ["Strategic candidates", 7, "Industrial logic, product adjacency, geography, regulatory and conflicts"],
    ["Structure fit", 19, "Ability to accept staged funding, 8% PIK, 1.0x preference and milestones"],
    ["Outreach gate", 0, "No contact until a new committee decision explicitly authorizes it"],
], columns=["module", "candidate_count", "required_analysis"])
display(next_product)


## 11. Optional governed write


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = VAULT / "Reports" / "Transaction Design Reproductions" / run_stamp
manifest = {
    "generated_at_utc": run_stamp,
    "recommendation": recommendation,
    "next_authority": next_authority,
    "decision_checks": decision_checks,
    "guardrails": guardrails,
    "recommended_option": preferred.to_dict(),
    "valuation_scenarios": valuations.to_dict(orient="records"),
    "investor_returns": returns.to_dict(orient="records"),
}

if WRITE_REPRODUCTION_ARTIFACTS:
    assert all(decision_checks.values())
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "loop_006_reproduction_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    structures.to_csv(output_dir / "transaction_structures.csv", index=False)
    valuations.to_csv(output_dir / "valuation_sensitivities.csv", index=False)
    returns.to_csv(output_dir / "investor_returns.csv", index=False)
    print("Wrote governed reproduction artifacts:", output_dir)
else:
    print("DRY RUN — the transaction decision was reproduced and the vault was not changed.")


## What Baby Step 6 demonstrates

The operating system can now transform a validated capital need into a governed transaction hypothesis:

**Funding need → structure alternatives → leverage/dilution controls → valuation sensitivities → investor-return election → approved internal working structure.**

The output is still not a transaction. It is a controlled modeling baseline that can be used in Baby Step 7 to rank synthetic counterparties before any outreach decision.
